# 03 - Chunking Performance Benchmark

Compare performance of different chunking modes:
- **memory_only**: Fastest, no DB writes
- **persist_selective**: Only persist failed evaluations
- **persist_all**: Full DB persistence (slowest)

## 1. Setup

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath(".."))
os.chdir("..")
print(f"Working directory: {os.getcwd()}")

In [ ]:
CATEGORY_ID = "8.1"  # Use a category with multiple files for meaningful benchmark
MARINE_DIR = os.path.join(os.getcwd(), "test_submissions", "marine")
JUROR_ROLE = "reasoning_model"
CONSENSUS_ROLE = "consensus_model"

## 2. Discover Test Category

In [ ]:
from engine.pipeline import discover_category_folders

categories = discover_category_folders(MARINE_DIR)
target = [c for c in categories if c["category_id"] == CATEGORY_ID]

if not target:
    print(f"ERROR: Category {CATEGORY_ID} not found")
else:
    folder_path = target[0]["path"]
    file_count = sum(len(files) for _, _, files in os.walk(folder_path))
    print(f"Test category: {target[0]["folder_name"]}")
    print(f"Files: {file_count}")

## 3. Benchmark Function

In [ ]:
from engine.pipeline import score_category
from engine.utils.eval_config import EvaluationConfig
from engine.utils.chunking_config import ChunkingMode

def benchmark_mode(mode: ChunkingMode, label: str) -> dict:
    config = EvaluationConfig(
        marine_dir=MARINE_DIR,
        juror_role=JUROR_ROLE,
        consensus_role=CONSENSUS_ROLE,
        chunking_mode=mode,
        persist_failed=True
    )
    
    print(f"
{"=" * 60}")
    print(f"Benchmarking: {label}")
    print(f"Mode: {mode.value}")
    print(f"{"=" * 60}")
    
    start = time.time()
    result = score_category(CATEGORY_ID, folder_path, silent=True, config=config)
    elapsed = time.time() - start
    
    return {
        "mode": mode.value,
        "label": label,
        "time_sec": round(elapsed, 2),
        "final_score": result.get("final_score", 0),
        "evidence_count": result.get("evidence_count", 0),
        "html_path": result.get("html_viewer_path", "")
    }

## 4. Run Benchmarks

In [ ]:
results = []

print("Starting benchmarks...")
print("Note: LLM calls may take 30-60s each")

# Benchmark 1: Memory Only (fastest)
results.append(benchmark_mode(ChunkingMode.MEMORY_ONLY, "Memory Only (No DB)"))

# Benchmark 2: Persist Selective
results.append(benchmark_mode(ChunkingMode.PERSIST_SELECTIVE, "Persist Selective (Failed Only)"))

# Benchmark 3: Persist All (slowest)
results.append(benchmark_mode(ChunkingMode.PERSIST_ALL, "Persist All (Full DB)"))

## 5. Results Comparison

In [ ]:
print("
" + "=" * 80)
print("BENCHMARK RESULTS")
print("=" * 80)
print(f"{"Mode":<30} {"Time (s)":<12} {"Score":<8} {"Files":<8} {"Speedup":<10}")
print("-" * 80)

baseline_time = results[-1]["time_sec"]  # Use persist_all as baseline

for r in results:
    speedup = baseline_time / r["time_sec"] if r["time_sec"] > 0 else 1.0
    print(f"{r["label"]:<30} {r["time_sec"]:<12.2f} {r["final_score"]:<8} {r["evidence_count"]:<8} {speedup:<10.2f}x")

print("-" * 80)
print(f"
Baseline (persist_all): {baseline_time:.2f}s")

## 6. Analysis

In [ ]:
memory_time = results[0]["time_sec"]
selective_time = results[1]["time_sec"]
all_time = results[2]["time_sec"]

print("Performance Analysis:")
print(f"  memory_only vs persist_all:    {all_time / memory_time:.2f}x faster")
print(f"  persist_selective vs persist_all: {all_time / selective_time:.2f}x faster")
print(f"
Recommendation:")
print(f"  - Use memory_only for: research, testing, batch evaluation")
print(f"  - Use persist_selective for: production with appeal/review needs")
print(f"  - Use persist_all for: full audit trail requirements")

## 7. Save Benchmark Results

In [ ]:
os.makedirs("notebook_v3/outputs", exist_ok=True)
out_path = "notebook_v3/outputs/chunking_benchmark.json"

benchmark_data = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "category_id": CATEGORY_ID,
    "file_count": file_count,
    "results": results
}

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_data, f, indent=2, ensure_ascii=False)

print(f"Benchmark saved to {out_path}")